# Editor de PDF con máxima fidelidad visual

Este notebook permite **reemplazar texto, eliminar páginas, insertar texto nuevo, agregar secciones, agregar/quitar hipervínculos y modificar metadatos** en un PDF local, con una interfaz gráfica basada en `ipywidgets`.

> **Requisito de versión:** `pdf-edit-engine` 0.2.0 requiere **Python 3.12 o superior**. Verifica tu versión con `python --version` antes de continuar.

Cada operación siempre trabaja sobre una **copia** del PDF (nunca sobre el original) y crea automáticamente un respaldo `.backup.pdf` en la misma carpeta.


In [7]:
# Instalación de dependencias
import sys
print(f"Versión de Python detectada: {sys.version}")
if sys.version_info < (3, 12):
    print("\n⚠ ADVERTENCIA: pdf-edit-engine requiere Python 3.12 o superior.")
    print("  Este notebook no funcionará correctamente con una versión anterior.")

# !pip install --quiet --upgrade pdf-edit-engine pikepdf ipywidgets pandas


Versión de Python detectada: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


In [2]:
# Importación de bibliotecas
from pathlib import Path
import shutil
import traceback

import pikepdf              # usado solo para contar páginas (ya es dependencia de pdf-edit-engine)
import pdf_edit_engine as pee   # motor real de edición de PDF con preservación de formato

import ipywidgets as widgets
from IPython.display import display, clear_output, FileLink

print("Bibliotecas cargadas correctamente.")
print("Funciones disponibles en pdf_edit_engine:",
      sorted(n for n in dir(pee) if not n.startswith("_") and callable(getattr(pee, n))))


Bibliotecas cargadas correctamente.
Funciones disponibles en pdf_edit_engine: ['Annotation', 'ContentElement', 'Degradation', 'DegradationKind', 'Edit', 'EditResult', 'EncodingError', 'FidelityReport', 'FontInfo', 'FontNotFoundError', 'GraphicsStateSnapshot', 'OperatorError', 'PDFEditError', 'Paragraph', 'ReflowError', 'TextBlock', 'TextCharacter', 'TextMatch', 'add_annotation', 'add_bookmark', 'add_highlight', 'add_hyperlink', 'add_watermark', 'analyze_subset', 'batch_replace', 'batch_replace_block', 'can_render', 'compute_uniform_layout', 'crop_pages', 'decrypt_pdf', 'delete_annotation', 'delete_block', 'delete_pages', 'detect_paragraphs', 'edit_metadata', 'encrypt_pdf', 'extend_subset', 'extract_bbox_text', 'fill_form', 'find', 'flatten_annotations', 'get_annotations', 'get_fonts', 'get_text', 'get_text_layout', 'insert_text_block', 'merge_pdfs', 'move_annotation', 'reflow_paragraph', 'reorder_pages', 'replace', 'replace_all', 'replace_block', 'rotate_pages', 'shift_content_below', 

In [3]:
# Funciones de utilidad: backup, estado del documento, extracción de estilo, reporte de fidelidad

WORKSPACE = Path("pdf_editor_workspace")
WORKSPACE.mkdir(exist_ok=True)

# Estado del documento actualmente cargado en la interfaz.
estado = {
    "nombre_base": None,     # nombre del archivo sin extensión
    "original_path": None,   # PDF tal como se subió (nunca se modifica)
    "backup_path": None,     # copia de seguridad .backup.pdf
    "current_path": None,    # última versión editada (o el original si aún no hay ediciones)
    "version": 0,            # contador de versiones generadas
}


def obtener_archivo_subido(upload_widget):
    """Obtiene (nombre, bytes) del FileUpload, compatible con ipywidgets 7.x y 8.x."""
    valor = upload_widget.value
    if not valor:
        return None, None
    if isinstance(valor, dict):  # ipywidgets 7.x: dict {nombre: info}
        nombre, info = next(iter(valor.items()))
        contenido = info["content"]
    else:  # ipywidgets 8.x: tupla de dicts
        info = valor[0]
        nombre = info["name"]
        contenido = info["content"]
    return nombre, bytes(contenido)


def cargar_pdf(nombre, contenido_bytes):
    """Guarda el PDF subido, crea una copia de seguridad .backup.pdf y reinicia el estado.

    El archivo original NUNCA se modifica: todas las ediciones se aplican sobre
    versiones nuevas (`_v1.pdf`, `_v2.pdf`, ...) generadas en WORKSPACE.
    """
    nombre_base = Path(nombre).stem
    original_path = WORKSPACE / f"{nombre_base}.pdf"
    original_path.write_bytes(contenido_bytes)

    backup_path = WORKSPACE / f"{nombre_base}.backup.pdf"
    shutil.copyfile(original_path, backup_path)

    estado.update({
        "nombre_base": nombre_base,
        "original_path": original_path,
        "backup_path": backup_path,
        "current_path": original_path,
        "version": 0,
    })
    return original_path


def contar_paginas(path):
    with pikepdf.open(str(path)) as pdf:
        return len(pdf.pages)


def nueva_ruta_version(sufijo=""):
    """Genera una ruta de salida única para la próxima versión editada del documento."""
    estado["version"] += 1
    extra = f"_{sufijo}" if sufijo else ""
    return WORKSPACE / f"{estado['nombre_base']}_v{estado['version']}{extra}.pdf"


def extraer_estilo(path, texto_referencia):
    """Busca `texto_referencia` en el PDF y devuelve su fuente, tamaño y color.

    Se usa para que el texto/las secciones nuevas hereden el estilo de un texto
    ya existente en el documento, en vez de usar valores fijos.
    """
    coincidencias = pee.find(str(path), texto_referencia)
    if not coincidencias:
        return None
    caracter = coincidencias[0].characters[0]
    return {
        "font_name": caracter.font_name,
        "font_size": caracter.font_size,
        "color": caracter.color,
    }


def mostrar_reporte_fidelidad(resultados):
    """Muestra el FidelityReport de una lista de EditResult de pdf_edit_engine."""
    filas = []
    for r in resultados:
        fr = r.fidelity_report
        filas.append({
            "éxito": r.success,
            "acción de fuente": r.font_action,
            "fuente preservada": fr.font_preserved,
            "desbordamiento": fr.overflow_detected,
            "reflujo aplicado": fr.reflow_applied,
            "glifos faltantes": ", ".join(fr.glyphs_missing) if fr.glyphs_missing else "—",
        })

    try:
        import pandas as pd
        display(pd.DataFrame(filas))
    except ImportError:
        for fila in filas:
            print(fila)

    for r in resultados:
        for d in r.fidelity_report.degradations:
            print(f"  ⚠ [{d.severity}] {d.kind}: {d.detail}")

print("Funciones de utilidad definidas.")


Funciones de utilidad definidas.


In [4]:
# Funciones de edición — una por operación solicitada.
# Todas trabajan sobre archivos en disco y devuelven (ruta_de_salida, info_de_resultado).

def reemplazar_texto(current_path, texto_buscar, texto_nuevo, output_path, reemplazar_todas=False, pagina=None):
    """Reemplaza texto preservando fuente, tamaño y posición (pdf_edit_engine.replace/replace_all)."""
    if not texto_buscar:
        raise ValueError("Debes indicar el texto a buscar.")

    if reemplazar_todas:
        resultados = pee.replace_all(str(current_path), texto_buscar, texto_nuevo, str(output_path))
        if not resultados:
            raise ValueError(f"No se encontró el texto '{texto_buscar}'.")
    else:
        coincidencias = pee.find(str(current_path), texto_buscar, page=pagina)
        if not coincidencias:
            extra = f" en la página {pagina + 1}." if pagina is not None else "."
            raise ValueError(f"No se encontró el texto '{texto_buscar}'{extra}")
        resultados = [pee.replace(str(current_path), coincidencias[0], texto_nuevo, str(output_path))]

    return output_path, resultados


def eliminar_paginas(current_path, paginas_1indexed, output_path):
    """Elimina las páginas indicadas (numeradas desde 1 en la interfaz)."""
    total = contar_paginas(current_path)
    for p in paginas_1indexed:
        if p < 1 or p > total:
            raise ValueError(f"La página {p} no existe (el documento tiene {total} páginas).")

    paginas_0indexed = [p - 1 for p in paginas_1indexed]
    pee.delete_pages(str(current_path), paginas_0indexed, str(output_path))
    restantes = contar_paginas(output_path)
    return output_path, {"eliminadas": paginas_1indexed, "paginas_restantes": restantes}


def insertar_texto(current_path, pagina, x, y, texto, output_path, texto_referencia=None, tamano=12.0):
    """Inserta texto nuevo en una posición exacta, opcionalmente copiando el estilo de un texto de referencia."""
    if not texto.strip():
        raise ValueError("El texto a insertar no puede estar vacío.")

    font_name = None
    font_size = tamano
    if texto_referencia:
        estilo = extraer_estilo(current_path, texto_referencia)
        if estilo is None:
            raise ValueError(f"No se encontró el texto de referencia '{texto_referencia}'.")
        font_name = estilo["font_name"]
        font_size = estilo["font_size"]

    resultado = pee.insert_text_block(
        str(current_path), pagina, x, y, texto, str(output_path),
        font_name=font_name, font_size=font_size,
    )
    return output_path, [resultado]


def agregar_seccion(current_path, pagina, x, y, titulo, contenido, output_path, generar_temporal,
                     ref_titulo=None, ref_contenido=None):
    """Agrega un título + contenido como una sección nueva, con estilos consistentes."""
    if not titulo.strip() or not contenido.strip():
        raise ValueError("El título y el contenido no pueden estar vacíos.")

    estilo_titulo = extraer_estilo(current_path, ref_titulo) if ref_titulo else None
    estilo_contenido = extraer_estilo(current_path, ref_contenido) if ref_contenido else None

    font_titulo = estilo_titulo["font_name"] if estilo_titulo else None
    size_titulo = estilo_titulo["font_size"] if estilo_titulo else 16.0
    font_contenido = estilo_contenido["font_name"] if estilo_contenido else None
    size_contenido = estilo_contenido["font_size"] if estilo_contenido else 11.0

    # Paso 1: título (a un archivo intermedio)
    paso_titulo = generar_temporal("titulo")
    res_titulo = pee.insert_text_block(
        str(current_path), pagina, x, y, titulo, str(paso_titulo),
        font_name=font_titulo, font_size=size_titulo,
    )

    # Paso 2: contenido, debajo del título, sobre el resultado del paso 1
    y_contenido = y - size_titulo - 14
    res_contenido = pee.insert_text_block(
        str(paso_titulo), pagina, x, y_contenido, contenido, str(output_path),
        font_name=font_contenido, font_size=size_contenido,
    )

    return output_path, [res_titulo, res_contenido]


def agregar_hipervinculo(current_path, modo, pagina, texto, url, output_path, generar_temporal,
                          x=None, y=None, tamano=12.0):
    """Agrega un hipervínculo, ya sea sobre texto existente o insertando texto nuevo enlazado.

    Nota: pdf_edit_engine no permite forzar color/subrayado en texto nuevo insertado;
    el texto nuevo hereda el estilo por defecto de la página. Si quieres un enlace con
    apariencia típica (azul, subrayado), usa el modo "texto existente" sobre un texto
    que ya tenga ese estilo, o edítalo después.
    """
    if not url.strip():
        raise ValueError("Indica la URL del hipervínculo.")

    if modo == "existente":
        if not texto.strip():
            raise ValueError("Indica el texto existente que se convertirá en enlace.")
        coincidencias = pee.find(str(current_path), texto, page=pagina)
        if not coincidencias:
            raise ValueError(f"No se encontró el texto '{texto}' en la página {pagina + 1}.")
        bbox = coincidencias[0].bounding_box
        pee.add_hyperlink(str(current_path), pagina, bbox, url, str(output_path))
        return output_path, f"Se convirtió el texto '{texto}' en un hipervínculo hacia {url}."

    else:  # modo == "nuevo"
        if not texto.strip():
            raise ValueError("Escribe el texto nuevo que se mostrará como enlace.")
        paso_texto = generar_temporal("texto_enlace")
        pee.insert_text_block(str(current_path), pagina, x, y, texto, str(paso_texto), font_size=tamano)

        coincidencias = pee.find(str(paso_texto), texto, page=pagina)
        if not coincidencias:
            raise ValueError("No se pudo localizar el texto recién insertado para crear el enlace.")
        bbox = coincidencias[-1].bounding_box
        pee.add_hyperlink(str(paso_texto), pagina, bbox, url, str(output_path))
        return output_path, f"Se insertó el texto '{texto}' como hipervínculo hacia {url}."


def quitar_hipervinculo(current_path, fragmento_url, generar_temporal):
    """Elimina todas las anotaciones de hipervínculo cuya URL contenga `fragmento_url`."""
    if not fragmento_url:
        raise ValueError("Indica un fragmento de la URL a eliminar.")

    ruta_trabajo = current_path
    eliminados = 0
    while True:
        anotaciones = pee.get_annotations(str(ruta_trabajo))
        objetivo = next((a for a in anotaciones if a.uri and fragmento_url in a.uri), None)
        if objetivo is None:
            break
        nueva_ruta = generar_temporal("sin_link")
        pee.delete_annotation(str(ruta_trabajo), objetivo, str(nueva_ruta))
        ruta_trabajo = nueva_ruta
        eliminados += 1

    if eliminados == 0:
        raise ValueError(f"No se encontraron hipervínculos que contengan '{fragmento_url}'.")

    return ruta_trabajo, eliminados


def modificar_metadatos(current_path, metadatos, output_path):
    """Modifica metadatos del documento (título, autor, asunto, palabras clave, ...)."""
    if not metadatos:
        raise ValueError("Indica al menos un campo de metadatos a modificar.")
    pee.edit_metadata(str(current_path), metadatos, str(output_path))
    return output_path, metadatos

print("Funciones de edición definidas.")


Funciones de edición definidas.


In [5]:
# Interfaz de usuario con ipywidgets

# --- Carga de archivo ---
upload = widgets.FileUpload(accept=".pdf", description="Subir PDF", multiple=False)
info_pdf = widgets.HTML(value="<i>Ningún PDF cargado todavía.</i>")

# --- Selector de operación ---
operacion = widgets.Dropdown(
    options=[
        ("Reemplazar texto", "reemplazar"),
        ("Eliminar páginas", "eliminar_paginas"),
        ("Insertar texto nuevo", "insertar_texto"),
        ("Agregar sección (título + contenido)", "agregar_seccion"),
        ("Agregar hipervínculo", "agregar_hipervinculo"),
        ("Quitar hipervínculo", "quitar_hipervinculo"),
        ("Modificar metadatos", "metadatos"),
    ],
    description="Operación:",
    style={"description_width": "initial"},
)

# --- Reemplazar texto ---
txt_buscar = widgets.Text(description="Buscar:", placeholder="Texto a reemplazar", style={"description_width": "initial"})
txt_nuevo = widgets.Text(description="Nuevo texto:", placeholder="Texto nuevo", style={"description_width": "initial"})
num_pagina_reemplazo = widgets.BoundedIntText(description="Página (0 = todas):", value=0, min=0, max=9999, style={"description_width": "initial"})
chk_todas = widgets.Checkbox(description="Reemplazar todas las coincidencias", value=False)
seccion_reemplazar = widgets.VBox([txt_buscar, txt_nuevo, num_pagina_reemplazo, chk_todas])

# --- Eliminar páginas ---
txt_paginas_eliminar = widgets.Text(description="Páginas:", placeholder="ej. 2,4,5 (numeración desde 1)", style={"description_width": "initial"})
seccion_eliminar = widgets.VBox([txt_paginas_eliminar])

# --- Insertar texto ---
num_pagina_insertar = widgets.BoundedIntText(description="Página:", value=1, min=1, max=9999, style={"description_width": "initial"})
num_x_insertar = widgets.FloatText(description="X:", value=72.0)
num_y_insertar = widgets.FloatText(description="Y:", value=700.0)
txt_insertar = widgets.Textarea(description="Texto:", placeholder="Texto a insertar", style={"description_width": "initial"})
txt_ref_insertar = widgets.Text(description="Ref. estilo (opcional):", placeholder="Texto existente del que copiar fuente/tamaño", style={"description_width": "initial"})
num_size_insertar = widgets.FloatText(description="Tamaño:", value=12.0)
seccion_insertar = widgets.VBox([num_pagina_insertar, num_x_insertar, num_y_insertar, txt_insertar, txt_ref_insertar, num_size_insertar])

# --- Agregar sección ---
num_pagina_seccion = widgets.BoundedIntText(description="Página:", value=1, min=1, max=9999, style={"description_width": "initial"})
num_x_seccion = widgets.FloatText(description="X:", value=72.0)
num_y_seccion = widgets.FloatText(description="Y (del título):", value=700.0, style={"description_width": "initial"})
txt_titulo_seccion = widgets.Text(description="Título:", style={"description_width": "initial"})
txt_contenido_seccion = widgets.Textarea(description="Contenido:", style={"description_width": "initial"})
txt_ref_titulo = widgets.Text(description="Ref. título (opcional):", placeholder="Texto de un título existente", style={"description_width": "initial"})
txt_ref_contenido = widgets.Text(description="Ref. contenido (opcional):", placeholder="Texto de un párrafo existente", style={"description_width": "initial"})
seccion_agregar_seccion = widgets.VBox([
    num_pagina_seccion, num_x_seccion, num_y_seccion,
    txt_titulo_seccion, txt_contenido_seccion, txt_ref_titulo, txt_ref_contenido,
])

# --- Agregar hipervínculo ---
modo_link = widgets.RadioButtons(
    options=[("Convertir texto ya existente en enlace", "existente"), ("Insertar texto nuevo como enlace", "nuevo")],
    description="Modo:",
)
num_pagina_link = widgets.BoundedIntText(description="Página:", value=1, min=1, max=9999, style={"description_width": "initial"})
txt_link_texto = widgets.Text(description="Texto:", placeholder="Texto existente o nuevo", style={"description_width": "initial"})
txt_link_url = widgets.Text(description="URL:", placeholder="https://...", style={"description_width": "initial"})
num_x_link = widgets.FloatText(description="X (solo si es nuevo):", value=72.0, style={"description_width": "initial"})
num_y_link = widgets.FloatText(description="Y (solo si es nuevo):", value=700.0, style={"description_width": "initial"})
num_size_link = widgets.FloatText(description="Tamaño (solo si es nuevo):", value=12.0, style={"description_width": "initial"})
seccion_hipervinculo = widgets.VBox([
    modo_link, num_pagina_link, txt_link_texto, txt_link_url, num_x_link, num_y_link, num_size_link,
])

# --- Quitar hipervínculo ---
txt_url_quitar = widgets.Text(description="URL contiene:", placeholder="fragmento de la URL a eliminar", style={"description_width": "initial"})
btn_listar_links = widgets.Button(description="Listar enlaces actuales")
seccion_quitar_link = widgets.VBox([txt_url_quitar, btn_listar_links])

# --- Metadatos ---
txt_meta_titulo = widgets.Text(description="Título:")
txt_meta_autor = widgets.Text(description="Autor:")
txt_meta_asunto = widgets.Text(description="Asunto:")
txt_meta_palabras = widgets.Text(description="Palabras clave:", style={"description_width": "initial"})
seccion_metadatos = widgets.VBox([txt_meta_titulo, txt_meta_autor, txt_meta_asunto, txt_meta_palabras])

secciones_por_operacion = {
    "reemplazar": seccion_reemplazar,
    "eliminar_paginas": seccion_eliminar,
    "insertar_texto": seccion_insertar,
    "agregar_seccion": seccion_agregar_seccion,
    "agregar_hipervinculo": seccion_hipervinculo,
    "quitar_hipervinculo": seccion_quitar_link,
    "metadatos": seccion_metadatos,
}

contenedor_dinamico = widgets.VBox([seccion_reemplazar])

def al_cambiar_operacion(change):
    contenedor_dinamico.children = [secciones_por_operacion[operacion.value]]

operacion.observe(al_cambiar_operacion, names="value")

# --- Controles generales ---
barra_progreso = widgets.IntProgress(min=0, max=100, value=0, description="Progreso:")
btn_ejecutar = widgets.Button(description="Ejecutar", button_style="primary")
btn_restaurar = widgets.Button(description="Restaurar original")
btn_descargar = widgets.Button(description="Descargar PDF editado", disabled=True, button_style="success")
salida = widgets.Output()


def on_upload_change(change):
    with salida:
        clear_output()
        try:
            nombre, contenido = obtener_archivo_subido(upload)
            if nombre is None:
                return
            ruta = cargar_pdf(nombre, contenido)
            n_paginas = contar_paginas(ruta)
            info_pdf.value = (
                f"<b>Archivo cargado:</b> {nombre} — {n_paginas} página(s).<br>"
                f"<b>Copia de seguridad:</b> <code>{estado['backup_path']}</code>"
            )
            btn_descargar.disabled = True
            print(f"PDF cargado correctamente en: {ruta}")
        except Exception as e:
            print("❌ Error al cargar el PDF:", e)
            traceback.print_exc()

upload.observe(on_upload_change, names="value")


def ejecutar_operacion(_boton):
    with salida:
        clear_output()
        barra_progreso.value = 0

        if estado["current_path"] is None:
            print("❌ Primero sube un PDF.")
            return
        if not Path(estado["current_path"]).exists():
            print("❌ El PDF cargado ya no está disponible en disco. Vuelve a subirlo.")
            return

        try:
            barra_progreso.value = 20
            op = operacion.value
            actual = estado["current_path"]

            if op == "reemplazar":
                pagina = None if num_pagina_reemplazo.value == 0 else num_pagina_reemplazo.value - 1
                salida_path = nueva_ruta_version()
                salida_path, resultados = reemplazar_texto(
                    actual, txt_buscar.value, txt_nuevo.value, salida_path,
                    reemplazar_todas=chk_todas.value, pagina=pagina,
                )
                barra_progreso.value = 80
                mostrar_reporte_fidelidad(resultados)

            elif op == "eliminar_paginas":
                texto_paginas = txt_paginas_eliminar.value.strip()
                if not texto_paginas:
                    raise ValueError("Indica al menos una página a eliminar.")
                paginas = [int(p.strip()) for p in texto_paginas.split(",") if p.strip()]
                salida_path = nueva_ruta_version()
                salida_path, info = eliminar_paginas(actual, paginas, salida_path)
                barra_progreso.value = 80
                print(f"Páginas eliminadas: {info['eliminadas']}")
                print(f"Páginas restantes: {info['paginas_restantes']}")

            elif op == "insertar_texto":
                pagina0 = num_pagina_insertar.value - 1
                ref = txt_ref_insertar.value.strip() or None
                salida_path = nueva_ruta_version()
                salida_path, resultados = insertar_texto(
                    actual, pagina0, num_x_insertar.value, num_y_insertar.value,
                    txt_insertar.value, salida_path, texto_referencia=ref, tamano=num_size_insertar.value,
                )
                barra_progreso.value = 80
                mostrar_reporte_fidelidad(resultados)

            elif op == "agregar_seccion":
                pagina0 = num_pagina_seccion.value - 1
                ref_t = txt_ref_titulo.value.strip() or None
                ref_c = txt_ref_contenido.value.strip() or None
                salida_path = nueva_ruta_version()
                salida_path, resultados = agregar_seccion(
                    actual, pagina0, num_x_seccion.value, num_y_seccion.value,
                    txt_titulo_seccion.value, txt_contenido_seccion.value, salida_path,
                    nueva_ruta_version, ref_titulo=ref_t, ref_contenido=ref_c,
                )
                barra_progreso.value = 80
                mostrar_reporte_fidelidad(resultados)

            elif op == "agregar_hipervinculo":
                pagina0 = num_pagina_link.value - 1
                salida_path = nueva_ruta_version()
                salida_path, mensaje = agregar_hipervinculo(
                    actual, modo_link.value, pagina0, txt_link_texto.value, txt_link_url.value,
                    salida_path, nueva_ruta_version,
                    x=num_x_link.value, y=num_y_link.value, tamano=num_size_link.value,
                )
                barra_progreso.value = 80
                print(mensaje)

            elif op == "quitar_hipervinculo":
                salida_path, eliminados = quitar_hipervinculo(actual, txt_url_quitar.value.strip(), nueva_ruta_version)
                barra_progreso.value = 80
                print(f"Se eliminaron {eliminados} hipervínculo(s) que contenían '{txt_url_quitar.value.strip()}'.")

            elif op == "metadatos":
                metadatos = {}
                if txt_meta_titulo.value.strip():
                    metadatos["title"] = txt_meta_titulo.value.strip()
                if txt_meta_autor.value.strip():
                    metadatos["author"] = txt_meta_autor.value.strip()
                if txt_meta_asunto.value.strip():
                    metadatos["subject"] = txt_meta_asunto.value.strip()
                if txt_meta_palabras.value.strip():
                    metadatos["keywords"] = txt_meta_palabras.value.strip()
                salida_path = nueva_ruta_version()
                salida_path, info = modificar_metadatos(actual, metadatos, salida_path)
                barra_progreso.value = 80
                print(f"Metadatos actualizados: {info}")

            else:
                raise ValueError(f"Operación desconocida: {op}")

            estado["current_path"] = salida_path
            btn_descargar.disabled = False
            barra_progreso.value = 100
            print(f"\n✅ Operación completada. Versión actual: {Path(salida_path).name}")

        except Exception as e:
            barra_progreso.value = 0
            print(f"❌ Error durante la edición: {e}")
            traceback.print_exc()

btn_ejecutar.on_click(ejecutar_operacion)


def restaurar_original(_boton):
    with salida:
        clear_output()
        if estado["original_path"] is None:
            print("❌ No hay ningún PDF cargado todavía.")
            return
        estado["current_path"] = estado["original_path"]
        estado["version"] = 0
        btn_descargar.disabled = True
        print("↺ Se restauró el documento a su versión original (sin ediciones).")
        print(f"Copia de seguridad disponible en: {estado['backup_path']}")

btn_restaurar.on_click(restaurar_original)


def descargar(_boton):
    with salida:
        clear_output()
        if estado["current_path"] is None:
            print("❌ No hay ningún archivo para descargar.")
            return
        ruta = Path(estado["current_path"])
        if not ruta.exists():
            print("❌ El archivo ya no existe en disco.")
            return
        print(f"Archivo listo ({ruta.name}):")
        display(FileLink(str(ruta), result_html_prefix="⬇ Haz clic para descargar: "))

btn_descargar.on_click(descargar)


def listar_links(_boton):
    with salida:
        clear_output()
        if estado["current_path"] is None:
            print("❌ Primero sube un PDF.")
            return
        try:
            anotaciones = pee.get_annotations(str(estado["current_path"]))
            enlaces = [a for a in anotaciones if a.subtype == "Link" and a.uri]
            if not enlaces:
                print("No se encontraron hipervínculos en la versión actual del documento.")
            for a in enlaces:
                print(f"Página {a.page + 1} | URL: {a.uri} | Rect: {a.rect}")
        except Exception as e:
            print("❌ Error al listar hipervínculos:", e)
            traceback.print_exc()

btn_listar_links.on_click(listar_links)

print("Interfaz definida. Ejecuta la siguiente celda para mostrarla.")


Interfaz definida. Ejecuta la siguiente celda para mostrarla.


In [6]:
# Mostrar la interfaz
display(widgets.VBox([
    widgets.HTML("<h3>1. Cargar PDF</h3>"),
    upload,
    info_pdf,
    widgets.HTML("<h3>2. Elegir operación</h3>"),
    operacion,
    contenedor_dinamico,
    widgets.HTML("<h3>3. Ejecutar</h3>"),
    widgets.HBox([btn_ejecutar, btn_restaurar, btn_descargar]),
    barra_progreso,
    salida,
]))


## Instrucciones de uso

1. **Sube tu PDF** con el botón de carga. Se creará automáticamente una copia `.backup.pdf` en la carpeta `pdf_editor_workspace/`, junto al archivo original — que nunca se modifica directamente.
2. **Elige la operación** en el menú desplegable; los campos se actualizarán según lo que necesites.
3. **Completa los campos** y presiona **Ejecutar**. Verás la barra de progreso y, al terminar, el reporte de fidelidad (para reemplazo/inserción de texto) o un resumen de lo realizado (para el resto de operaciones).
4. Puedes **encadenar varias ediciones**: cada operación se aplica sobre el resultado de la anterior (versiones `_v1.pdf`, `_v2.pdf`, ... en `pdf_editor_workspace/`).
5. Si algo sale mal, usa **Restaurar original** para volver al PDF tal como lo subiste (sin perder el backup).
6. Cuando el resultado te convenza, presiona **Descargar PDF editado** para obtener el enlace de descarga del archivo actual.

### Notas sobre fidelidad y limitaciones reales de `pdf_edit_engine`
- El **reemplazo de texto** y la **inserción de texto/secciones** preservan la fuente original siempre que sea posible; el reporte de fidelidad indica si hubo sustitución de fuente, desbordamiento, reflujo de párrafo o glifos faltantes.
- **Insertar texto nuevo o convertirlo en hipervínculo** no permite fijar manualmente el color o el subrayado del texto: hereda el estilo tipográfico de la página. Para un enlace con apariencia clásica (azul y subrayado), usa el modo "convertir texto existente" sobre un texto que ya tenga ese estilo.
- Si el PDF es **escaneado** (sin texto seleccionable), `find`/`replace` no encontrarán coincidencias porque no hay texto real en el contenido; en ese caso se necesita OCR previo (fuera del alcance de este notebook).
- Las coordenadas `X`/`Y` están en puntos PDF, con el origen (0,0) en la esquina **inferior izquierda** de la página.
